# Final Thesis I experiments (Kaggle, unattended)

- **Part A** (inverse scattering, about 2.5 h): 2D with sparsity on 5 cells, incl. a *fair* pixel baseline with the same sparsity term; 3D on real CytoPacq cells (BBBC024).
- **Part B** (4D Navier-Stokes, about 4–5 h): MLP vs Mamba vs bidirectional Mamba, with temporal smoothness.

**Settings:** Accelerator = **GPU T4 x2** (or P100), **Internet = On** → **Save Version → Save & Run All (Commit)**.
Output: **`final_results.zip`** (Output tab), with one log per run in `logs/`.

In [ ]:
REPO_URL = "https://github.com/muhammadhussainqureshi/thesis-gs"   # change after transferring the repo
PARTS = ["A", "B"]          # ["A"] or ["B"] to run only one part
QUICK = False               # True = 2-minute smoke test with tiny settings

import os, subprocess, sys, glob, zipfile, shutil, torch
WORK = "/kaggle/working"
CODE = "/kaggle/working/thesis-gs"
DATA = f"{WORK}/data"
OUT = f"{WORK}/final"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU")
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.isfile(f"{CODE}/bin/run_experiments.py"), "Repo has no bin/run_experiments.py - push the new structure first"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "tifffile"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Real 3D cells: BBBC024 ground-truth masks (30 .ics volumes, 20 HL60 nuclei each, made with CytoPacq)
VOL = "/tmp/cytopacq"
os.makedirs(VOL, exist_ok=True)
name = "BBBC024_v1_c00_highSNR_foreground.zip"
zp = f"{VOL}/{name}"
if "A" in PARTS:
    try:
        if not os.path.exists(zp):
            subprocess.run(["wget", "-q", "-O", zp, f"https://data.broadinstitute.org/bbbc/BBBC024/{name}"])
        zipfile.ZipFile(zp).extractall(VOL)
        print(len(glob.glob(f"{VOL}/**/*.ics", recursive=True)), "BBBC024 volumes")
    except Exception as e:
        print("BBBC024 download failed (3D runs will be skipped):", e)

In [ ]:
extra = ["--quick"] if QUICK else []
for part in PARTS:
    cfg = f"{CODE}/configs/final_part_{part.lower()}.json"
    cmd = [sys.executable, f"{CODE}/bin/run_experiments.py", cfg, "--out", f"{OUT}/part_{part.lower()}",
           "--data_dir", DATA, "--volumes", VOL] + extra
    subprocess.run(cmd)        # failures are logged per run; the notebook keeps going

In [ ]:
import pandas as pd
from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUT}/**/summary.csv", recursive=True)):
    print("\n=====", os.path.relpath(os.path.dirname(f), OUT))
    display(pd.read_csv(f))
    png = os.path.join(os.path.dirname(f), "reconstructions.png")
    if os.path.exists(png):
        display(Image(filename=png))
os.makedirs(OUT, exist_ok=True)
shutil.make_archive(f"{WORK}/final_results", "zip", OUT)
print("\nDownload final_results.zip from the Output tab.")